# RBF Isopleth Surface Builder

This notebook demonstrates how to transform isopleth monitoring data into a smooth surface using `scipy.interpolate.Rbf`. Update the configuration cell to target different datasets or interpolation behaviors, then re-run the processing and plotting cells to evaluate the results.

## How To Use
1. Adjust the `CONFIG` dictionary to point at the dataset and tweak interpolation/plotting parameters.
2. Run the *Load & Prepare Data* cell to pull a clean set of latitude/longitude/value triples.
3. Execute the *Interpolate With RBF* cell to build a surface grid.
4. Render the *2D Contour* and *3D Surface* plots to verify the fit against the raw points.

In [ ]:
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.path import Path as MplPath
from matplotlib.lines import Line2D
import matplotlib.ticker as ticker
from scipy.interpolate import RBFInterpolator
from scipy.spatial import KDTree, ConvexHull
from scipy.stats import gaussian_kde

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
CONFIG = {
    # Path to the CSV containing isopleth data. Update as needed.
    'data_path': Path('../data/utah/filtered_by_clearing_index/SR_clearing_nonmax_spring_1000.csv'),
    # Toggle saving artifacts for the current run.
    'save': False,
    # Directory where plots and params.csv will be written when save=True.
    'output_dir': Path('../plots/utah/SR_filtered/Clearing_Index_Filtered/1000/Non_max_spring/RBF'),
    # Column names in the CSV that hold latitude, longitude, and the value to interpolate.
    'lat_col': 'NOx',
    'lon_col': 'VOC',
    'value_col': 'Ozone',
    # Optional filters applied before interpolation, e.g. {'parameter': ['Carbon disulfide']}.
    'filters': {},
    # Optional: Filter out data points with VOC above this threshold. Set to None to disable.\n
    'voc_max_threshold': 900,
    # Number of grid steps (per axis) when evaluating the surface. Higher = smoother but slower.
    'grid_resolution': 2000,
    # Parameters forwarded directly to scipy.interpolate.RBFInterpolator.
    'rbf': {
        'kernel': 'gaussian',
        'epsilon':   0.011009009009009009,
        'smoothing': 0.7007007007007008,
    },
    # Plot styling options.
    'plot': {
        'show_point_values': False,
        'cmap': 'viridis',
        'contour_levels': 50,
        'colorbar_limits': (55, 61),
        'colorbar_step': 1, #TODO: Change this back to 1
        'surface_alpha': 0.75,
        'scatter_color': 'k',
        'plotly_colorscale': 'Viridis',
        'point_size': 15,
        'font_size': 14,
    },
    # Mask settings: keep only regions supported by at least N neighbors within a normalized radius.
    'mask': {
        'min_neighbors': 2,        # Require this many neighbors within radius (including the point itself).
        'neighbor_radius': 0.2,   #0.2 If None, auto-computed on normalized coordinates; otherwise interpreted in normalized units.
    },
    'title': 'Radial basis function-Non max weekend clearing index under 1000',
}
CONFIG


In [ ]:
# Helper to build an RBFInterpolator instance
def make_rbf_interpolator(lon_array, lat_array, value_array, rbf_kwargs):
    lon_array = np.asarray(lon_array, dtype=float)
    lat_array = np.asarray(lat_array, dtype=float)
    value_array = np.asarray(value_array, dtype=float)

    if lon_array.shape != lat_array.shape:
        raise ValueError('Longitude and latitude arrays must share the same shape.')

    normalized = dict(rbf_kwargs or {})
    if 'function' in normalized and 'kernel' not in normalized:
        normalized['kernel'] = normalized.pop('function')
    if 'smooth' in normalized and 'smoothing' not in normalized:
        normalized['smoothing'] = normalized.pop('smooth')

    sample_points = np.column_stack((lon_array, lat_array))
    return RBFInterpolator(sample_points, value_array, **normalized)

In [ ]:
# Mask grid cells to the convex hull of points that have enough nearby neighbors
# Distances are computed after normalizing x/y ranges so radius works across mixed scales.
def _dense_convex_hull_mask(x_points, y_points, grid_x, grid_y, min_neighbors=2, neighbor_radius=None):
    x_points = np.asarray(x_points, dtype=float)
    y_points = np.asarray(y_points, dtype=float)
    points = np.column_stack((x_points, y_points))

    if len(points) < 3 or min_neighbors <= 1:
        return np.ones_like(grid_x, dtype=bool)

    # Normalize coordinates so distance radius is scale-agnostic.
    span_x = np.ptp(x_points)
    span_y = np.ptp(y_points)
    scale_x = span_x if span_x > 0 else 1.0
    scale_y = span_y if span_y > 0 else 1.0
    norm_points = np.column_stack((x_points / scale_x, y_points / scale_y))

    tree = KDTree(norm_points)
    radius = neighbor_radius
    if radius is None:
        k = min(max(int(min_neighbors), 2), len(points))
        distances, _ = tree.query(norm_points, k=k)
        kth = distances[:, -1]
        radius = np.percentile(kth, 75)
        if radius == 0:
            radius = 0.05

    neighbor_counts = np.array([len(tree.query_ball_point(pt, r=radius)) for pt in norm_points])
    dense_points = points[neighbor_counts >= min_neighbors]
    if len(dense_points) < 3:
        return np.ones_like(grid_x, dtype=bool)

    hull = ConvexHull(dense_points)
    hull_path = MplPath(dense_points[hull.vertices])
    flat_grid = np.column_stack((grid_x.ravel(), grid_y.ravel()))
    mask = hull_path.contains_points(flat_grid, radius=1e-9)
    return mask.reshape(grid_x.shape)


In [ ]:
# Load & Prepare Data
data_path = Path(CONFIG['data_path'])
lat_col = CONFIG['lat_col']
lon_col = CONFIG['lon_col']
value_col = CONFIG['value_col']

SAVE_OUTPUTS = bool(CONFIG.get('save', False))
output_dir_config = CONFIG.get('output_dir') or (data_path.parent / 'rbf_outputs')
OUTPUT_DIR = Path(output_dir_config).expanduser()
if not OUTPUT_DIR.is_absolute():
    OUTPUT_DIR = Path.cwd() / OUTPUT_DIR
RUN_ID = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_PREFIX = f'rbf_surface_{RUN_ID}'

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(data_path)

for col, allowed in CONFIG['filters'].items():
    if allowed:
        df = df[df[col].isin(allowed)]

voc_limit = CONFIG.get('voc_max_threshold')
if voc_limit is not None and 'VOC' in df.columns:
    # Coerce to numeric for threshold comparison, keeping rows <= voc_limit
    voc_numeric = pd.to_numeric(df['VOC'], errors='coerce')
    df = df[voc_numeric <= voc_limit]
required_surface_cols = ['Ozone', 'VOC', 'NOx']
missing_surface_cols = [col for col in required_surface_cols if col not in df.columns]
if missing_surface_cols:
    raise ValueError(f"Missing required columns for surface generation: {missing_surface_cols}")
rows_before_surface_filter = len(df)
df = df.dropna(subset=required_surface_cols).copy()
dropped_surface_rows = rows_before_surface_filter - len(df)
if dropped_surface_rows:
    print(f"Removed {dropped_surface_rows} rows with null surface inputs (Ozone/VOC/NOx).")

needed_cols = [lat_col, lon_col, value_col]
df = df[needed_cols].copy()

for col in needed_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=needed_cols)

if len(df) < 4:
    raise ValueError('Need at least four points for RBF interpolation. Adjust filters or dataset.')

print(f"Loaded {len(df)} usable points from {data_path}")
df.head()

In [ ]:
# Interpolate With RBFInterpolator
lon = df[lon_col].to_numpy()
lat = df[lat_col].to_numpy()
values = df[value_col].to_numpy()

# --- NEW LOGIC START: Determine grid extent based on dense points ---
mask_cfg = CONFIG.get('mask', {}) or {}
min_neighbors = int(mask_cfg.get('min_neighbors', 1))
neigh_radius = mask_cfg.get('neighbor_radius')
if neigh_radius is not None:
    neigh_radius = float(neigh_radius)

# We basically repeat the logic from _dense_convex_hull_mask to find the "dense" subset
# so we can bound the grid to that area.
points = np.column_stack((lon, lat))
if len(points) < 3 or min_neighbors <= 1:
    # Fallback to full extent if filtering isn't applicable
    use_lon_min, use_lon_max = lon.min(), lon.max()
    use_lat_min, use_lat_max = lat.min(), lat.max()
    radius_val = 0.0 # Placeholder
else:
    # Reuse normalization logic for consistent radius interpretation
    span_x = np.ptp(lon)
    span_y = np.ptp(lat)
    scale_x = span_x if span_x > 0 else 1.0
    scale_y = span_y if span_y > 0 else 1.0
    norm_points = np.column_stack((lon / scale_x, lat / scale_y))

    tree = KDTree(norm_points)
    radius_val = neigh_radius
    if radius_val is None:
        k = min(max(int(min_neighbors), 2), len(points))
        # distances to kth nearest neighbor
        distances, _ = tree.query(norm_points, k=k)
        kth = distances[:, -1]
        # Auto-radius: 75th percentile of kth-neighbor distance
        radius_val = np.percentile(kth, 75)
        if radius_val == 0:
            radius_val = 0.05

    # Identify which points have enough neighbors within radius
    neighbor_counts = np.array([len(tree.query_ball_point(pt, r=radius_val)) for pt in norm_points])
    dense_mask = neighbor_counts >= min_neighbors
    dense_points = points[dense_mask]

    if len(dense_points) < 3:
        # Fallback if too few dense points
        use_lon_min, use_lon_max = lon.min(), lon.max()
        use_lat_min, use_lat_max = lat.min(), lat.max()
    else:
        # Use extent of dense points + a small margin
        d_lon = dense_points[:, 0]
        d_lat = dense_points[:, 1]
        
        # Add a margin (e.g. 5% of the dense span)
        margin_x = np.ptp(d_lon) * 0.05
        margin_y = np.ptp(d_lat) * 0.05
        
        use_lon_min, use_lon_max = d_lon.min() - margin_x, d_lon.max() + margin_x
        use_lat_min, use_lat_max = d_lat.min() - margin_y, d_lat.max() + margin_y

# --- NEW LOGIC END ---

grid_res = CONFIG['grid_resolution']
lon_lin = np.linspace(use_lon_min, use_lon_max, grid_res)
lat_lin = np.linspace(use_lat_min, use_lat_max, grid_res)
lon_grid, lat_grid = np.meshgrid(lon_lin, lat_lin)

rbf = make_rbf_interpolator(lon, lat, values, CONFIG['rbf'])
grid_points = np.column_stack((lon_grid.ravel(), lat_grid.ravel()))
value_grid = rbf(grid_points).reshape(lon_grid.shape)

hull_mask = _dense_convex_hull_mask(
    lon,
    lat,
    lon_grid,
    lat_grid,
    min_neighbors=min_neighbors,
    neighbor_radius=neigh_radius,
)
value_grid = np.where(hull_mask, value_grid, np.nan)

value_min = float(np.nanmin(value_grid))
value_max = float(np.nanmax(value_grid))
value_mean = float(np.nanmean(value_grid))

SURFACE_VALUE_MAX = value_max

summary = {
    'value_min': value_min,
    'value_max': value_max,
    'value_mean': value_mean,
}

norm = colors.Normalize(vmin=0, vmax=value_max)

params_record = {
    'run_id': RUN_ID,
    'data_path': str(data_path),
    'lat_col': lat_col,
    'lon_col': lon_col,
    'value_col': value_col,
    'grid_resolution': grid_res,
    'filters': json.dumps(CONFIG.get('filters', {})),
    'value_min': value_min,
    'value_max': value_max,
    'value_mean': value_mean,
}
for key, value in CONFIG.get('rbf', {}).items():
    params_record[f'rbf_{key}'] = value
for key, value in CONFIG.get('plot', {}).items():
    params_record[f'plot_{key}'] = value

if SAVE_OUTPUTS:
    params_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_params.csv'
    pd.DataFrame([params_record]).to_csv(params_path, index=False)

summary


In [ ]:
# Cross-Validation Fit Check
def evaluate_rbf_kfold(lon_array, lat_array, value_array, rbf_kwargs, k=5, shuffle=True, random_state=42):
    lon_array = np.asarray(lon_array)
    lat_array = np.asarray(lat_array)
    value_array = np.asarray(value_array)
    n_samples = len(value_array)

    if k < 2 or k > n_samples:
        raise ValueError('k must be at least 2 and no larger than the number of samples.')

    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)

    folds = np.array_split(indices, k)
    fold_stats = []

    for fold_id, test_idx in enumerate(folds, start=1):
        train_idx = np.setdiff1d(indices, test_idx, assume_unique=True)
        if len(train_idx) < 4:
            raise ValueError('Each training fold needs at least four points. Reduce k or use more data.')

        rbf = make_rbf_interpolator(
            lon_array[train_idx],
            lat_array[train_idx],
            value_array[train_idx],
            rbf_kwargs,
        )
        test_points = np.column_stack((lon_array[test_idx], lat_array[test_idx]))
        preds = rbf(test_points)
        residuals = preds - value_array[test_idx]

        fold_stats.append({
            'fold': fold_id,
            'n_test': int(len(test_idx)),
            'rmse': float(np.sqrt(np.mean(residuals ** 2))),
            'mae': float(np.mean(np.abs(residuals))),
            'bias': float(np.mean(residuals)),
        })

    results = pd.DataFrame(fold_stats)
    summary = {
        'rmse_mean': float(results['rmse'].mean()),
        'mae_mean': float(results['mae'].mean()),
        'bias_mean': float(results['bias'].mean()),
    }
    return results, summary

cv_results, cv_summary = evaluate_rbf_kfold(
    lon, lat, values, CONFIG['rbf'],
    k=5,
    shuffle=True,
    random_state=42,
)

display(cv_results)
cv_summary

In [ ]:
# 2D Contour Plot
plot_cfg = CONFIG['plot']
base_fs = plot_cfg.get('font_size', 12)
fig, ax = plt.subplots(figsize=(8, 6))

vmin = float(np.round(value_min-1))
vmax = float(np.round(value_max+1))
norm = colors.Normalize(vmin=vmin, vmax=vmax)

level_step = plot_cfg.get('colorbar_step')
levels_cfg = plot_cfg.get('contour_levels', 15)
use_step = level_step is not None and float(level_step) > 0
if use_step:
    step = float(level_step)
    levels = np.arange(norm.vmin, norm.vmax + step, step)
elif isinstance(levels_cfg, int):
    levels = np.linspace(norm.vmin, norm.vmax, levels_cfg)
else:
    levels = levels_cfg

contour = ax.contourf(lon_grid, lat_grid, value_grid, levels=levels, cmap=plot_cfg['cmap'], norm=norm)
show_point_values = plot_cfg.get('show_point_values', True)
scatter_kwargs = {
    'edgecolor': 'white',
    'linewidth': 0.5,
    's': plot_cfg['point_size'],
    'norm': norm,
}
if show_point_values:
    scatter_kwargs['c'] = values
    scatter_kwargs['cmap'] = plot_cfg['cmap']
else:
    scatter_kwargs['c'] = 'black'

scatter = ax.scatter(lon, lat, **scatter_kwargs)

# Mean point marker
mean_lon = float(np.nanmean(lon))
mean_lat = float(np.nanmean(lat))
mean_handle = ax.scatter(
    mean_lon,
    mean_lat,
    marker='X',
    s=160,
    color='white',
    edgecolor='black',
    linewidth=1.2,
    zorder=5,
)

# KDE-based 95% density contour (black outline)
kde_handle = None
kde = gaussian_kde(np.vstack([lon, lat]))
density = kde(np.vstack([lon_grid.ravel(), lat_grid.ravel()]))
density_grid = density.reshape(lon_grid.shape)
if np.all(np.isfinite(density_grid)):
    sorted_idx = np.argsort(density_grid.ravel())[::-1]
    cumulative = np.cumsum(density_grid.ravel()[sorted_idx])
    cumulative /= cumulative[-1]
    cutoff_idx = np.searchsorted(cumulative, 0.95)
    cutoff_density = density_grid.ravel()[sorted_idx[min(cutoff_idx, len(sorted_idx) - 1)]]
    kde_contours = ax.contour(
        lon_grid,
        lat_grid,
        density_grid,
        levels=[cutoff_density],
        colors='black',
        linewidths=1.4,
    )
    # Matplotlib versions differ on available attributes; use a manual legend handle.
    kde_handle = Line2D([0], [0], color='black', linewidth=1.4)

cbar = fig.colorbar(contour, ax=ax)
cbar.set_label("Ozone (ppb)", fontsize=base_fs+2, fontweight='bold')
cbar.ax.tick_params(labelsize=base_fs)
plt.setp(cbar.ax.get_yticklabels(), fontweight='bold')

if use_step:
    cbar.set_ticks(np.arange(norm.vmin, norm.vmax + step, step))

ax.set_xlabel("VOC (ppbC)", fontsize=base_fs+2, fontweight='bold')
ax.set_ylabel("NO$_x$ (ppb)", fontsize=base_fs+2, fontweight='bold')
# ax.set_title(CONFIG['title'])

ax.grid(False)
ax.tick_params(axis='both', which='major', labelsize=base_fs, bottom=True, left=True)
plt.setp(ax.get_xticklabels(), fontweight='bold')
plt.setp(ax.get_yticklabels(), fontweight='bold')

handles = [scatter, mean_handle]
labels = ['Observed points', 'Mean VOC/NOx']
if kde_handle is not None:
    handles.append(kde_handle)
    labels.append('95% KDE extent')
#ax.legend(handles, labels, loc='best')
ax.xaxis.set_major_locator(ticker.MaxNLocator(6))
ax.yaxis.set_major_locator(ticker.MaxNLocator(6))
# Set axis limits based on masked surface extent + margin
valid_mask = np.isfinite(value_grid)
if np.any(valid_mask):
    valid_lon = lon_grid[valid_mask]
    valid_lat = lat_grid[valid_mask]
    min_x, max_x = valid_lon.min(), valid_lon.max()
    min_y, max_y = valid_lat.min(), valid_lat.max()
    margin_x = (max_x - min_x) * 0.05
    margin_y = (max_y - min_y) * 0.05
    ax.set_xlim(min_x - margin_x, max_x + margin_x)
    ax.set_ylim(min_y - margin_y, max_y + margin_y)
else:
    ax.set_xlim(lon_grid.min(), lon_grid.max())
    ax.set_ylim(lat_grid.min(), lat_grid.max())
if SAVE_OUTPUTS:
    fig.savefig(OUTPUT_DIR / f'{RUN_ID}_2d_contour.png', dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# 3D Surface Plot
plot_cfg = CONFIG['plot']
base_fs = plot_cfg.get('font_size', 12)
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 - imported for side effects

vmin = float(np.round(value_min))
vmax = float(np.round(value_max))
norm = colors.Normalize(vmin=vmin, vmax=vmax)

level_step = plot_cfg.get('colorbar_step')
use_step = level_step is not None and float(level_step) > 0
step = float(level_step) if use_step else None

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(lon_grid, lat_grid, value_grid, cmap=plot_cfg['cmap'], norm=norm, alpha=plot_cfg['surface_alpha'])
show_point_values = plot_cfg.get('show_point_values', True)
scatter_3d_kwargs = {
    's': plot_cfg['point_size'],
    'depthshade': False
}
if show_point_values:
    scatter_3d_kwargs['c'] = values
    scatter_3d_kwargs['cmap'] = plot_cfg['cmap']
else:
    scatter_3d_kwargs['color'] = 'black'

ax.scatter(lon, lat, values, **scatter_3d_kwargs)

ax.set_xlabel(lon_col, fontsize=base_fs+2, fontweight='bold')
ax.set_ylabel(lat_col, fontsize=base_fs+2, fontweight='bold')
ax.set_zlabel(value_col, fontsize=base_fs+2, fontweight='bold')
# ax.set_title(CONFIG['title'])

ax.grid(False)
ax.tick_params(axis='both', which='major', labelsize=base_fs)
for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
    for label in axis.get_ticklabels():
        label.set_fontweight('bold')

cbar = fig.colorbar(surface, ax=ax, shrink=0.5, aspect=12)
cbar.set_label(value_col, fontsize=base_fs+2, fontweight='bold')
cbar.ax.tick_params(labelsize=base_fs)
plt.setp(cbar.ax.get_yticklabels(), fontweight='bold')

if use_step:
    cbar.set_ticks(np.arange(norm.vmin, norm.vmax + step, step))

if SAVE_OUTPUTS:
    fig.savefig(OUTPUT_DIR / f'{RUN_ID}_3d_surface.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Interactive 3D Plot (Plotly)
try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError('Plotly is required for the interactive 3D view. Install it with `pip install plotly`.') from exc

plotly_scale = CONFIG['plot'].get('plotly_colorscale') or plot_cfg['cmap'].capitalize()
base_fs = CONFIG['plot'].get('font_size', 12)
marker_size = max(plot_cfg['point_size'] / 4, 4)
vmin = float(np.round(np.nanmin(value_grid)))
vmax = float(np.round(np.nanmax(value_grid)))


# Downsample for interactions if grid is huge (e.g. > 250x250) to prevent browser hang
max_dim = 250
stride = max(1, int(max(lon_grid.shape) / max_dim))
if stride > 1:
    print(f'Downsampling Plotly surface by {stride}x (original {lon_grid.shape}) for smoothness.')
    p_lon = lon_grid[::stride, ::stride]
    p_lat = lat_grid[::stride, ::stride]
    p_val = value_grid[::stride, ::stride]
else:
    p_lon, p_lat, p_val = lon_grid, lat_grid, value_grid

fig = go.Figure()
fig.add_surface(
    x=p_lon,
    y=p_lat,
    z=p_val,
    colorscale=plotly_scale,
    opacity=plot_cfg['surface_alpha'],
    cmin=vmin,
    cmax=vmax,
    showscale=True,
    colorbar=dict(
        title=dict(text=f"<b>{value_col}</b>", font=dict(size=base_fs+2)),
        tickfont=dict(size=base_fs, family='Arial Black'),
    ),
    name='RBF surface',
)
show_point_values = CONFIG['plot'].get('show_point_values', True)
if show_point_values:
    marker_color = values
    marker_scale = plotly_scale
else:
    marker_color = 'black'
    marker_scale = None

fig.add_scatter3d(
    x=lon,
    y=lat,
    z=values,
    mode='markers',
    marker=dict(
        size=marker_size,
        color=marker_color,
        colorscale=marker_scale,
        line=dict(width=1, color='white'),
        cmin=vmin,
        cmax=vmax,
    ),
    name='Observed points',
)
fig.update_layout(
    # title='RBF Surface - Interactive 3D',
    scene=dict(
        xaxis=dict(
            title=dict(text=f"<b>{lon_col}</b>", font=dict(size=base_fs+2)),
            tickfont=dict(size=base_fs, family='Arial Black'),
            showgrid=False, ticks='outside'
        ),
        yaxis=dict(
            title=dict(text=f"<b>{lat_col}</b>", font=dict(size=base_fs+2)),
            tickfont=dict(size=base_fs, family='Arial Black'),
            showgrid=False, ticks='outside'
        ),
        zaxis=dict(
            title=dict(text=f"<b>{value_col}</b>", font=dict(size=base_fs+2)),
            tickfont=dict(size=base_fs, family='Arial Black'),
            showgrid=False, ticks='outside'
        ),
    ),
    margin=dict(l=0, r=0, t=10, b=0),
)

if SAVE_OUTPUTS:
    interactive_path = OUTPUT_DIR / f'{ARTIFACT_PREFIX}_interactive.html'
    fig.write_html(interactive_path, include_plotlyjs='cdn')

fig.show()

In [ ]:

# Hyperparameter Grid Search (RMSE minimization)
from itertools import product


def _unique_preserve(values):
    seen = set()
    ordered = []
    for item in values:
        if item not in seen:
            ordered.append(item)
            seen.add(item)
    return ordered


def grid_search_rbf_params(lon_array, lat_array, value_array, param_grid, **cv_kwargs):
    lon_array = np.asarray(lon_array, dtype=float)
    lat_array = np.asarray(lat_array, dtype=float)
    value_array = np.asarray(value_array, dtype=float)

    param_names = list(param_grid)
    combos = list(product(*param_grid.values()))
    rows = []

    for combo in combos:
        candidate = dict(zip(param_names, combo))
        try:
            _, summary = evaluate_rbf_kfold(
                lon_array,
                lat_array,
                value_array,
                candidate,
                **cv_kwargs,
            )
            rows.append({
                **candidate,
                'rmse_mean': summary['rmse_mean'],
                'mae_mean': summary['mae_mean'],
                'bias_mean': summary['bias_mean'],
                'status': 'ok',
            })
        except Exception as exc:
            rows.append({
                **candidate,
                'rmse_mean': np.inf,
                'mae_mean': np.inf,
                'bias_mean': np.nan,
                'status': f'failed: {exc}',
            })

    results = pd.DataFrame(rows)
    if not results.empty:
        results = results.sort_values(
            'rmse_mean',
            key=lambda values: np.where(np.isfinite(values), values, np.inf),
        ).reset_index(drop=True)
    return results


def progressive_refinement_grid_search(lon_array, lat_array, value_array, search_cfg, **cv_kwargs):
    categorical_grid = dict(search_cfg.get('categorical_params') or {})
    numeric_cfg = dict(search_cfg.get('numeric_params') or {})
    top_k = int(search_cfg.get('top_k', 5))
    padding_fraction = float(search_cfg.get('padding_fraction', 0.25))
    max_iterations = int(search_cfg.get('max_iterations', 5))

    numeric_state = {}
    for name, meta in numeric_cfg.items():
        bounds = meta.get('bounds')
        if not bounds or len(bounds) != 2:
            raise ValueError(f'Missing 2-element bounds for numeric param {name!r}.')
        lower, upper = map(float, bounds)
        if lower >= upper:
            raise ValueError(f'Lower bound must be < upper bound for {name!r}.')
        current = meta.get('initial_range', bounds)
        current_lower, current_upper = map(float, current)
        current_lower = max(lower, current_lower)
        current_upper = min(upper, current_upper)
        if current_upper <= current_lower:
            current_lower, current_upper = lower, upper
        numeric_state[name] = {
            'bounds': (lower, upper),
            'current': (current_lower, current_upper),
            'points': max(2, int(meta.get('points', 10))),
            'min_span': float(meta.get('min_span', 1e-4)),
        }

    history_records = []
    combined_results = []
    best_overall_row = None
    best_overall_rmse = np.inf

    for iteration in range(1, max_iterations + 1):
        iteration_ranges = {name: tuple(state['current']) for name, state in numeric_state.items()}
        param_grid = {**categorical_grid}
        for name, state in numeric_state.items():
            low, high = iteration_ranges[name]
            param_grid[name] = np.linspace(low, high, state['points'])

        iter_results = grid_search_rbf_params(
            lon_array,
            lat_array,
            value_array,
            param_grid,
            **cv_kwargs,
        )

        if iter_results.empty:
            history_records.append({
                'iteration': iteration,
                'best_rmse': np.nan,
                **{f'{name}_min': rng[0] for name, rng in iteration_ranges.items()},
                **{f'{name}_max': rng[1] for name, rng in iteration_ranges.items()},
            })
            break

        iter_results = iter_results.assign(iteration=iteration)
        combined_results.append(iter_results)

        finite = iter_results[np.isfinite(iter_results['rmse_mean'])]
        if finite.empty:
            history_records.append({
                'iteration': iteration,
                'best_rmse': np.nan,
                **{f'{name}_min': rng[0] for name, rng in iteration_ranges.items()},
                **{f'{name}_max': rng[1] for name, rng in iteration_ranges.items()},
            })
            break

        best_row = finite.iloc[0]
        best_rmse = float(best_row['rmse_mean'])
        if best_rmse < best_overall_rmse:
            best_overall_rmse = best_rmse
            best_overall_row = best_row

        history_records.append({
            'iteration': iteration,
            'best_rmse': best_rmse,
            **{f'{name}_min': rng[0] for name, rng in iteration_ranges.items()},
            **{f'{name}_max': rng[1] for name, rng in iteration_ranges.items()},
        })


        if top_k <= 0:
            continue

        top_slice = finite.head(top_k)
        for name, state in numeric_state.items():
            lower_bound, upper_bound = state['bounds']
            span = iteration_ranges[name][1] - iteration_ranges[name][0]
            if span <= state['min_span']:
                continue

            top_min = float(top_slice[name].min())
            top_max = float(top_slice[name].max())
            if not np.isfinite(top_min) or not np.isfinite(top_max):
                continue

            padding = max(span * padding_fraction, state['min_span'])
            new_lower = max(lower_bound, top_min - padding)
            new_upper = min(upper_bound, top_max + padding)
            if new_upper - new_lower < state['min_span']:
                midpoint = 0.5 * (new_lower + new_upper)
                half_span = state['min_span'] / 2
                new_lower = max(lower_bound, midpoint - half_span)
                new_upper = min(upper_bound, midpoint + half_span)
            numeric_state[name]['current'] = (float(new_lower), float(new_upper))

    combined_df = pd.concat(combined_results, ignore_index=True) if combined_results else pd.DataFrame()
    if not combined_df.empty:
        combined_df = combined_df.sort_values(
            'rmse_mean',
            key=lambda values: np.where(np.isfinite(values), values, np.inf),
        ).reset_index(drop=True)

    history_df = pd.DataFrame(history_records)
    best_params = best_overall_row.to_dict() if best_overall_row is not None else {}
    if 'iteration' in best_params:
        best_params.pop('iteration')
    return {
        'history': history_df,
        'results': combined_df,
        'best_params': best_params,
    }


def _build_progressive_search_cfg(default_rbf):
    kernels = _unique_preserve([
        default_rbf.get('kernel', 'thin_plate_spline'),
        'thin_plate_spline',
        'cubic',
        'multiquadric',
        'gaussian',
        'linear',
        'inverse_multiquadric',
        'inverse_quadratic',
    ])
    return {
        'categorical_params': {
            'kernel': kernels,
        },
        'numeric_params': {
            'epsilon': {
                'bounds': (0.00001, 10), # 10 If it is overfitting it seems useful to make the upper bound for this lower
                'points': 40,
                'min_span': 5e-4,
            },
            'smoothing': {
                'bounds': (0.001, 100), # 0.001
                'points': 100,
                'min_span': 1e-4,
            },
        },
        'top_k': 10,
        'padding_fraction': 0.25,
        'max_iterations': 5,
    }


cv_settings = dict(k=5, shuffle=True, random_state=42)
progressive_cfg = _build_progressive_search_cfg(CONFIG.get('rbf', {}))
progressive_results = progressive_refinement_grid_search(
    lon,
    lat,
    values,
    progressive_cfg,
    **cv_settings,
)

history_df = progressive_results['history']
if not history_df.empty:
    print('Progressive refinement summary (ranges reflect the grid used each iteration):')
    display(history_df)
else:
    print('No history recorded; the refinement search did not complete any iterations.')

rbf_search_results = progressive_results['results']
print('Top candidate RBF settings across all iterations (sorted by mean RMSE):')
if not rbf_search_results.empty:
    display(rbf_search_results.head(10))
else:
    display(rbf_search_results)

best_rbf_params = progressive_results['best_params']
if best_rbf_params:
    print('Best RBF parameters based on progressive search:')
    best_rbf_params
else:
    print('No successful parameter combinations were found. Inspect rbf_search_results for diagnostics.')


In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import ParameterGrid

# 1. Define range of hyperparameters
param_grid = {
    'kernel': ['gaussian'],
    # A few distinct epsilon values (width of the RBF)
    'epsilon': np.linspace(0.001, 10, 1000),
    # A few distinct smoothing values (0 = exact fit, higher = more regularization)
    'smoothing': np.linspace(0.0, 100.0, 1000),
}

# 2. Helper for Spatial Grid Search CV
def spatial_grid_search_cv(lon_arr, lat_arr, val_arr, grid_dict, n_splits=5, random_state=42):
    """
    Performs grid search using spatial K-Means clustering for folds.
    This effectively tests the model's ability to extrapolate to "held-out regions"
    rather than just random points, punishing overfitting more realistically.
    """
    lon_arr = np.asarray(lon_arr)
    lat_arr = np.asarray(lat_arr)
    val_arr = np.asarray(val_arr)
    points = np.column_stack((lon_arr, lat_arr))
    
    # Create spatial folds using K-Means on coordinates
    kmeans = KMeans(n_clusters=n_splits, random_state=random_state, n_init='auto')
    fold_labels = kmeans.fit_predict(points)
    
    # Prepare list of parameter combinations
    params_list = list(ParameterGrid(grid_dict))
    
    results = []
    
    print(f"Starting Spatial Grid Search over {len(params_list)} combinations with {n_splits} spatial folds...")
    
    for params in params_list:
        fold_rmses = []
        fold_maes = []
        
        for fold_id in range(n_splits):
            # Train on all clusters except fold_id
            train_mask = (fold_labels != fold_id)
            test_mask = (fold_labels == fold_id)
            
            # Skip if split is degenerate
            if np.sum(train_mask) < 4 or np.sum(test_mask) < 1:
                continue
            
            train_lon = lon_arr[train_mask]
            train_lat = lat_arr[train_mask]
            train_val = val_arr[train_mask]
            
            test_points = points[test_mask]
            test_val = val_arr[test_mask]
            
            # Fit RBF
            rbf_model = make_rbf_interpolator(train_lon, train_lat, train_val, params)
            
            # Predict
            preds = rbf_model(test_points)
            residuals = preds - test_val
            
            rmse = np.sqrt(np.mean(residuals**2))
            mae = np.mean(np.abs(residuals))
            
            fold_rmses.append(rmse)
            fold_maes.append(mae)
        
        # Aggregate results for this parameter set
        if len(fold_rmses) > 0:
            mean_rmse = np.mean(fold_rmses)
            mean_mae = np.mean(fold_maes)
            results.append({
                'params': params,
                'rmse': mean_rmse,
                'mae': mean_mae
            })
    
    # Convert to DataFrame and sort
    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df = results_df.sort_values(by='rmse', ascending=True).reset_index(drop=True)
        
    return results_df

# 3. Optimize
spatial_cv_results = spatial_grid_search_cv(
    lon, lat, values, param_grid, n_splits=5, random_state=42
)

# 4. Display results
print("\nTop 5 Hyperparameter Combinations (Spatial CV):")
display(spatial_cv_results.head(5))

best_params = spatial_cv_results.iloc[0]['params'] if not spatial_cv_results.empty else None
print(f"\nBest Params found: {best_params}")
